<a href="https://colab.research.google.com/github/Moquiuti/LangGraph_Orquestrando_agentes_e_multiagentes/blob/main/Persist%C3%AAncia_e_streaming_com_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title
!pip install -q -U langgraph
!pip install -q -U langgraph-checkpoint-sqlite
!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-google-genai
!pip install -q -U langchain-tavily
!pip install -q -U python-dotenv

In [3]:
import os
import sqlite3
from typing import TypedDict, Annotated, Sequence

from dotenv import load_dotenv

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage
)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

In [4]:
try:
    from google.colab import userdata

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

    print("API Keys carregadas pelos Secrets do Colab.")

except Exception:
    load_dotenv()
    print("Tentando carregar API Keys via arquivo .env.")

API Keys carregadas pelos Secrets do Colab.


In [5]:
if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY não encontrada.")

if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY não encontrada.")

print("Chaves carregadas com sucesso.")

Chaves carregadas com sucesso.


In [6]:
#Criar o estado do agente
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

search_tool = TavilySearch(
    max_results=3,
    topic="general"
)

tools = [search_tool]

In [8]:
#classe Agent com LangGraph
class Agent:
    def __init__(self, model, tools, checkpointer):
        self.model = model.bind_tools(tools)
        self.tools = {tool.name: tool for tool in tools}

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_llm)
        graph.add_node("action", self.take_action)

        graph.set_entry_point("llm")

        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {
                True: "action",
                False: END
            }
        )

        graph.add_edge("action", "llm")

        self.graph = graph.compile(checkpointer=checkpointer)

    def call_llm(self, state: AgentState):
        messages = state["messages"]

        response = self.model.invoke(messages)

        return {
            "messages": [response]
        }

    def exists_action(self, state: AgentState):
        result = state["messages"][-1]

        return hasattr(result, "tool_calls") and len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]

            print(f"\nChamando ferramenta: {tool_name}")
            print(f"Argumentos: {tool_args}")

            if tool_name not in self.tools:
                result = f"Ferramenta {tool_name} não encontrada."
            else:
                result = self.tools[tool_name].invoke(tool_args)

            results.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    name=tool_name,
                    content=str(result)
                )
            )

        return {
            "messages": results
        }

In [9]:
#Configurando o SQlite
conn = sqlite3.connect(
    "memoria_langgraph.sqlite",
    check_same_thread=False
)

memory = SqliteSaver(conn)

In [10]:
#Instanciando o agente
agent = Agent(
    model=llm,
    tools=tools,
    checkpointer=memory
)

In [11]:
def executar_com_memoria(pergunta: str, thread_id: str = "conversa_1"):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    resultado = agent.graph.invoke(
        {
            "messages": [HumanMessage(content=pergunta)]
        },
        config=config
    )

    resposta_final = resultado["messages"][-1].content

    print("\n=== RESPOSTA FINAL ===")
    print(resposta_final)

    return resultado

In [12]:
resultado_1 = executar_com_memoria(
    "Meu nome é Leandro e eu estou estudando LangGraph.",
    thread_id="leandro"
)


=== RESPOSTA FINAL ===
Olá Leandro! É um prazer conhecê-lo. LangGraph é uma biblioteca muito interessante para construir agentes e fluxos de trabalho com LLMs. Se tiver alguma dúvida ou precisar de ajuda com LangGraph, é só perguntar!


In [13]:
resultado_2 = executar_com_memoria(
    "Qual é o meu nome e o que eu estou estudando?",
    thread_id="leandro"
)


=== RESPOSTA FINAL ===
Seu nome é Leandro e você está estudando LangGraph.


In [14]:
resultado_3 = executar_com_memoria(
    "Qual é o meu nome e o que eu estou estudando?",
    thread_id="usuario_teste"
)


=== RESPOSTA FINAL ===
[{'type': 'text', 'text': 'Não tenho acesso a informações pessoais, como seu nome ou o que você está estudando.', 'extras': {'signature': 'Co4DAQw51scjgWSArG9IECOqLELr3gmwu3z5dPFyEw3WW+74mDspLkrnQA9kD2mIKejmq34BWK7SLCtstqvoXTpFI9zSCwKu6m451CtVPdo9TgTB+cEmbMaYlwsgflbSc3UyBYDGMaAI9miL7ar6ZGF/9L7BiED4y4tBR+v+3XbNCheJSnVWJsEwqHz83ADNmWVmw31vMm4JagB/8u7sZoVILSxSHFuQKtvuH3XpbDGvjMCYwwtk4Z4X9eadYebAnPTr8/6W2+kRONFMsq8LmqSK4RN1Ehm71gKjgUZ5h/0QNB0NrkRRwvNi2Cn6FDRVhqFZATBfiAoapItviJRx3+4w9q/wlFsoImc8+L9dYpoyRZRRd3nWEcgADso1oZwWaf9MhA4+9sri1F/e9eIy+h+0sg3pFJLZa8GkXxSs1NGqxXpGNUqYHKJuTQCh/EXgcn3hCuvBa1pEu1QQOY89FgSaBih6kTYjhRe8d0Ez6tMAsMRvDkQ/hwWmoM9wfAZVyzLth3acXFKIGLojF2gPuYI='}}]


In [15]:
resultado_4 = executar_com_memoria(
    "Pesquise na internet o que é LangGraph e me explique de forma resumida.",
    thread_id="leandro"
)


Chamando ferramenta: tavily_search
Argumentos: {'search_depth': 'basic', 'query': 'what is LangGraph'}

=== RESPOSTA FINAL ===
[{'type': 'text', 'text': 'LangGraph é um framework de código aberto, construído sobre o LangChain, que simplifica a criação e o gerenciamento de fluxos de trabalho para agentes de IA. Ele utiliza arquiteturas baseadas em grafos para mapear, organizar e otimizar como os agentes de IA interagem e tomam decisões, combinando modelos de linguagem grandes (LLMs) com essas estruturas. Em resumo, LangGraph oferece uma maneira escalável e transparente de projetar sistemas de IA avançados, desde chatbots simples até sistemas multiagentes complexos.', 'extras': {'signature': 'CqkFAQw51sdOwbzRZ5WjYKaOKI/CpHwLbDuU0FlqBTjHVtTz89nmsCgRO9kulKqfVKOXMEySCMYZKFUtTpSIgX/B5qb6gQKeOAPSNOP1mO2fi54ufDsqJgoylHdb+VmrKzAkidOiOvufDHhnhxARytHByfO5HLv/EBy4GgS2xxG1S0566J445lMe/W3tpmMkCuIGaeDMAVNYdRqxS9Dhjl+HJFn78FBKa6TwwmDvRxUW5MF0x2CbWt3glN5vt0XoIjwvneSQwfKZfmf6zVGDk0KnIx63pKD83HFk8FpWKSD

In [16]:
def executar_stream(pergunta: str, thread_id: str = "conversa_stream"):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    eventos = agent.graph.stream(
        {
            "messages": [HumanMessage(content=pergunta)]
        },
        config=config,
        stream_mode="values"
    )

    ultima_resposta = None

    for evento in eventos:
        mensagens = evento.get("messages", [])

        if not mensagens:
            continue

        ultima_msg = mensagens[-1]

        print("\n--- EVENTO DO GRAFO ---")
        print("Tipo:", ultima_msg.type)

        if hasattr(ultima_msg, "tool_calls") and ultima_msg.tool_calls:
            print("Tool calls:", ultima_msg.tool_calls)

        print("Conteúdo:")
        print(ultima_msg.content)

        ultima_resposta = ultima_msg

    return ultima_resposta

In [17]:
executar_stream(
    "Pesquise as principais novidades recentes sobre agentes de IA e resuma em português.",
    thread_id="leandro_stream"
)


--- EVENTO DO GRAFO ---
Tipo: human
Conteúdo:
Pesquise as principais novidades recentes sobre agentes de IA e resuma em português.

--- EVENTO DO GRAFO ---
Tipo: ai
Tool calls: [{'name': 'tavily_search', 'args': {'time_range': 'week', 'topic': 'news', 'query': 'AI agents recent news'}, 'id': 'c0b0404d-1a02-436b-8a59-972abcd56b6e', 'type': 'tool_call'}]
Conteúdo:


Chamando ferramenta: tavily_search
Argumentos: {'time_range': 'week', 'topic': 'news', 'query': 'AI agents recent news'}

--- EVENTO DO GRAFO ---
Tipo: tool
Conteúdo:
{'query': 'AI agents recent news', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.cjr.org/tow_center/ai-agents-are-coming-for-news-can-publishers-reclaim-control.php', 'title': 'AI Agents Are Coming for News. Can Publishers Reclaim Control?', 'content': 'AI platforms are releasing agentic tools like ChatGPT Pulse and Huxe, which generate personalized news briefings based on information the platforms have stored', 'sc

AIMessage(content=[{'type': 'text', 'text': 'Aqui estão as principais novidades recentes sobre agentes de IA:\n\n*   **Agentes de IA e Notícias**: Plataformas de IA estão lançando ferramentas como ChatGPT Pulse e Huxe, que geram resumos de notícias personalizados. Isso levanta questões sobre como os editores de notícias podem manter o controle sobre seu conteúdo.\n*   **Notion como Hub para Agentes de IA**: O Notion lançou uma nova plataforma para desenvolvedores que permite a integração de agentes de IA, fontes de dados externas e código personalizado diretamente em seus espaços de trabalho.\n*   **Riscos dos Agentes de IA**: Pesquisas recentes indicam que agentes de IA automatizados podem se fixar perigosamente na conclusão de tarefas, sem reconhecer quando suas ações podem ser prejudiciais.', 'extras': {'signature': 'Cu8HAQw51sfvCZ0ftfUJB/64hZyPyhP34vod9mXbzf89fCRQpcFFznnKlcdeysrtazgfuqmquxdQRm40FtT5HOP650QRxv+2Owy2lc35vIaLIo+KKP57TEgy26pEPgJX9FDiguttgUhRFcM1D1cZ3pJjvIgx6j02BnJhZifn

In [18]:
executar_stream(
    "Com base no que você acabou de pesquisar, quais aplicações práticas isso teria para um desenvolvedor backend Java?",
    thread_id="leandro_stream"
)


--- EVENTO DO GRAFO ---
Tipo: human
Conteúdo:
Com base no que você acabou de pesquisar, quais aplicações práticas isso teria para um desenvolvedor backend Java?

--- EVENTO DO GRAFO ---
Tipo: ai
Conteúdo:
[{'type': 'text', 'text': 'Para um desenvolvedor backend Java, as novidades sobre agentes de IA teriam várias aplicações práticas e considerações:\n\n1.  **Integração com Plataformas de Agentes de IA:**\n    *   **APIs para Conteúdo Personalizado:** Se sua empresa lida com conteúdo (notícias, produtos, etc.), você pode precisar desenvolver APIs em Java para alimentar dados para agentes de IA que geram resumos ou recomendações personalizadas (como o ChatGPT Pulse). Isso envolveria a exposição de dados de forma segura e eficiente.\n    *   **Conectores para Hubs de Agentes:** Com plataformas como o Notion se tornando hubs para agentes de IA, um desenvolvedor Java pode ser responsável por criar conectores ou SDKs para integrar sistemas backend existentes (escritos em Java) com esses hub

AIMessage(content=[{'type': 'text', 'text': 'Para um desenvolvedor backend Java, as novidades sobre agentes de IA teriam várias aplicações práticas e considerações:\n\n1.  **Integração com Plataformas de Agentes de IA:**\n    *   **APIs para Conteúdo Personalizado:** Se sua empresa lida com conteúdo (notícias, produtos, etc.), você pode precisar desenvolver APIs em Java para alimentar dados para agentes de IA que geram resumos ou recomendações personalizadas (como o ChatGPT Pulse). Isso envolveria a exposição de dados de forma segura e eficiente.\n    *   **Conectores para Hubs de Agentes:** Com plataformas como o Notion se tornando hubs para agentes de IA, um desenvolvedor Java pode ser responsável por criar conectores ou SDKs para integrar sistemas backend existentes (escritos em Java) com esses hubs. Isso permitiria que os agentes de IA acessassem dados ou disparassem ações em seus sistemas.\n\n2.  **Automação de Fluxos de Trabalho e Processos Internos:**\n    *   **Orquestração de 

In [19]:
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = cursor.fetchall()

tabelas

[('checkpoints',), ('writes',)]

In [20]:
for tabela in tabelas:
    nome_tabela = tabela[0]
    print("\nTabela:", nome_tabela)

    try:
        cursor.execute(f"SELECT * FROM {nome_tabela} LIMIT 3;")
        linhas = cursor.fetchall()

        for linha in linhas:
            print(linha)

    except Exception as erro:
        print("Não foi possível ler tabela:", erro)


Tabela: checkpoints
('leandro', '', '1f14f9b6-d10c-6825-bfff-a561206e6e90', None, 'msgpack', b'\x87\xa1v\x04\xa2ts\xd9 2026-05-14T13:47:19.353726+00:00\xa2id\xd9$1f14f9b6-d10c-6825-bfff-a561206e6e90\xaechannel_values\x81\xa9__start__\x81\xa8messages\x91\xc7\xb9\x05\x94\xbdlangchain_core.messages.human\xacHumanMessage\x86\xa7content\xd93Meu nome \xc3\xa9 Leandro e eu estou estudando LangGraph.\xb1additional_kwargs\x80\xb1response_metadata\x80\xa4type\xa5human\xa4name\xc0\xa2id\xc0\xb3model_validate_json\xb0channel_versions\x81\xa9__start__\xd9300000000000000000000000000000001.0.9708793169209847\xadversions_seen\x81\xa9__input__\x80\xb0updated_channels\x91\xa9__start__', b'{"source": "input", "step": -1, "parents": {}}')
('leandro', '', '1f14f9b6-d117-6e0b-8000-d82b9d3b747f', '1f14f9b6-d10c-6825-bfff-a561206e6e90', 'msgpack', b'\x87\xa1v\x04\xa2ts\xd9 2026-05-14T13:47:19.358381+00:00\xa2id\xd9$1f14f9b6-d117-6e0b-8000-d82b9d3b747f\xaechannel_values\x82\xa8messages\x91\xc7\xde\x05\x94\xbd

In [21]:
from google.colab import files

files.download("memoria_langgraph.sqlite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>